In [2]:
import pyspark
from pyspark.sql import SparkSession # for creating a Spark session
from pyspark.sql import types # for defining the schema of the data
from pyspark.sql import functions as F # for using built-in functions
import pandas as pd


In [3]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName("test") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 17:29:25 WARN Utils: Your hostname, JACK2000-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.158 instead (on interface en0)
26/06/17 17:29:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 17:29:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Load the dataset

In [10]:
!curl -L -o data/raw/yellow/2025/fhvhv_tripdata_2021-01.csv.gz https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  123M  100  123M    0     0  10.4M      0  0:00:11  0:00:11 --:--:-- 13.4M


In [11]:
# Unzip
!gunzip data/raw/yellow/2025/fhvhv_tripdata_2021-01.csv.gz

In [12]:
!wc -l data/raw/yellow/2025/fhvhv_tripdata_2021-01.csv

 11908469 data/raw/yellow/2025/fhvhv_tripdata_2021-01.csv


In [13]:
df = spark.read.csv("data/raw/yellow/2025/fhvhv_tripdata_2021-01.csv", header=True)

In [14]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [15]:
# Extract the head (sample) of data via terminal command and save it to a new file
!head -n 1001 data/raw/yellow/2025/fhvhv_tripdata_2021-01.csv > data/raw/yellow/2025/head.csv

In [16]:
df_pandas = pd.read_csv("data/raw/yellow/2025/head.csv")

In [17]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

Create Spark Schema

In [18]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [19]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [20]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("data/raw/yellow/2025/fhvhv_tripdata_2021-01.csv")

Using Repartition

In [21]:
df = df.repartition(24)

In [25]:
df.write.parquet("data/processed/fhvhv/2021/01/")

26/06/17 17:54:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/17 17:54:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/17 17:54:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/17 17:54:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/17 17:54:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/17 17:54:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/17 17:54:55 WARN MemoryManager: Total allocation exceeds 95.00%

In [26]:
df = spark.read.parquet("data/processed/fhvhv/2021/01/")

In [27]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [28]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-01 07:26:54|2021-01-01 07:51:41|         225|         218|   NULL|
|           HV0003|              B02835|2021-01-02 08:38:56|2021-01-02 08:45:38|          42|          41|   NULL|
|           HV0005|              B02510|2021-01-02 07:15:12|2021-01-02 07:41:50|          21|         258|   NULL|
|           HV0005|              B02510|2021-01-02 07:40:46|2021-01-02 08:01:50|          81|         265|   NULL|
|           HV0005|              B02510|2021-01-01 05:52:11|2021-01-01 06:37:27|         239|         132|   NULL|
|           HV0003|              B02866|2021-01-02 20:31:06|2021-01-02 20:47:26|

## User-Defined Functions

Let's say we have a function that performs complex logic, something not easy to express with SQL. I'll call this function crazy_stuff.

For example, suppose it processes a column called dispatching_base_number. The logic could be:

- Extracts the numeric part of the string by removing the first character (base_num[1:]) and converts it to an integer (num).

- If the number is divisible by 7, return an ID starting with "S" followed by the number in hexadecimal format.

- If the number is divisible by 3, return an ID starting with "A" followed by the number in hexadecimal format.

- Otherwise, return an ID starting with "E" followed by the number in hexadecimal format.

In [30]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [31]:
crazy_stuff('B02884')

's/b44'

In [32]:
# User Define Function (UDF)
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [33]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
  .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
  .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
  .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
  .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-01|  2021-01-01|         225|         218|
|  s/b13| 2021-01-02|  2021-01-02|          42|          41|
|  e/9ce| 2021-01-02|  2021-01-02|          21|         258|
|  e/9ce| 2021-01-02|  2021-01-02|          81|         265|
|  e/9ce| 2021-01-01|  2021-01-01|         239|         132|
|  e/b32| 2021-01-02|  2021-01-02|          75|          42|
|  e/9ce| 2021-01-01|  2021-01-01|         247|          14|
|  e/b3b| 2021-01-02|  2021-01-02|           7|          17|
|  e/9ce| 2021-01-03|  2021-01-03|         107|           1|
|  e/b3b| 2021-01-01|  2021-01-01|         160|          28|
|  e/b48| 2021-01-01|  2021-01-01|         216|         216|
|  e/9ce| 2021-01-03|  2021-01-03|         237|          50|
|  e/b3b| 2021-01-01|  2021-01-01|         171|         131|
|  e/b3b| 2021-01-03|  2

The equivelent of using SQL in pyspark

SELECT * FROM df WHERE hvfhs_license_num = HV0003

In [36]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003') \
  .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 08:38:56|2021-01-02 08:45:38|          42|          41|
|2021-01-02 20:31:06|2021-01-02 20:47:26|          75|          42|
|2021-01-02 09:10:13|2021-01-02 09:28:27|           7|          17|
|2021-01-01 05:51:06|2021-01-01 06:03:30|         160|          28|
|2021-01-01 04:45:49|2021-01-01 04:51:08|         216|         216|
|2021-01-01 19:06:15|2021-01-01 19:25:52|         171|         131|
|2021-01-03 13:14:10|2021-01-03 13:27:10|         181|          66|
|2021-01-03 23:34:02|2021-01-03 23:39:56|         193|         179|
|2021-01-03 19:45:34|2021-01-03 20:08:39|         249|          37|
|2021-01-04 10:01:31|2021-01-04 10:13:58|          79|         237|
|2021-01-04 10:50:12|2021-01-04 11:13:37|         148|          68|
|2021-01-02 09:31:24|2021-01-02 09:35:26|       